# Chapter 03: Combining DataFrames

Data often lives in multiple tables that need to be combined before analysis. Pandas provides several functions for this: `concat` for stacking, and `merge` / `join` for database-style operations.

In this notebook we will cover:

- `pd.concat()` (axis=0, axis=1, keys, ignore_index)
- `pd.merge()` / `.merge()` — inner, outer, left, right joins
- Merge on different column names (`left_on`, `right_on`)
- Merge on index (`left_index`, `right_index`)
- `.join()` shorthand
- Handling duplicate column names (`suffixes`)

In [ ]:
import pandas as pd
import numpy as np

## Setting Up Example DataFrames

In [ ]:
# Two DataFrames with the same columns (like monthly sales reports)
jan = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Doohickey'],
    'units_sold': [150, 200, 80],
    'revenue': [4500, 8000, 1600]
})

feb = pd.DataFrame({
    'product': ['Widget', 'Gadget', 'Thingamajig'],
    'units_sold': [180, 170, 95],
    'revenue': [5400, 6800, 2850]
})

print("January:")
print(jan)
print("\nFebruary:")
print(feb)

## `pd.concat()` — Stacking DataFrames

### Vertical stacking (axis=0)

Appends rows from one DataFrame below another. Columns are aligned by name.

In [ ]:
combined = pd.concat([jan, feb])
combined

In [ ]:
# Notice the duplicate index values (0, 1, 2 appear twice)
# Use ignore_index=True to reset
combined = pd.concat([jan, feb], ignore_index=True)
combined

In [ ]:
# Use keys to add a hierarchical index identifying the source
combined = pd.concat([jan, feb], keys=['January', 'February'])
combined

In [ ]:
# Access a specific group using the hierarchical index
combined.loc['January']

### Horizontal stacking (axis=1)

Places DataFrames side by side. Rows are aligned by index.

In [ ]:
names = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie']}, index=[0, 1, 2])
scores = pd.DataFrame({'score': [92, 85, 78]}, index=[0, 1, 2])

pd.concat([names, scores], axis=1)

In [ ]:
# When indices do not align, NaN fills the gaps
a = pd.DataFrame({'x': [1, 2]}, index=[0, 1])
b = pd.DataFrame({'y': [10, 20]}, index=[1, 2])

pd.concat([a, b], axis=1)

## `pd.merge()` — Database-style Joins

Merge combines DataFrames based on shared column values, similar to SQL JOINs.

| Join type | Keeps |
|-----------|-------|
| `inner` | Only rows with matching keys in **both** DataFrames (default) |
| `outer` | All rows from **both**, filling NaN where no match |
| `left` | All rows from the **left** DataFrame |
| `right` | All rows from the **right** DataFrame |

In [ ]:
customers = pd.DataFrame({
    'customer_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'city': ['Sydney', 'Melbourne', 'Brisbane', 'Perth']
})

orders = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'customer_id': [1, 2, 2, 3, 5],
    'amount': [250, 130, 470, 90, 310]
})

print("Customers:")
print(customers)
print("\nOrders:")
print(orders)

In [ ]:
# Inner join: only customers who have orders AND orders from known customers
pd.merge(customers, orders, on='customer_id', how='inner')

In [ ]:
# Left join: all customers, with order info where available
pd.merge(customers, orders, on='customer_id', how='left')

In [ ]:
# Right join: all orders, with customer info where available
pd.merge(customers, orders, on='customer_id', how='right')

In [ ]:
# Outer join: everything from both sides
pd.merge(customers, orders, on='customer_id', how='outer')

## Merging on Different Column Names

When the join key has different names in each DataFrame, use `left_on` and `right_on`.

In [ ]:
employees = pd.DataFrame({
    'emp_id': [1, 2, 3],
    'name': ['Alice', 'Bob', 'Charlie'],
    'dept_code': ['ENG', 'MKT', 'ENG']
})

departments = pd.DataFrame({
    'code': ['ENG', 'MKT', 'FIN'],
    'department': ['Engineering', 'Marketing', 'Finance']
})

pd.merge(employees, departments, left_on='dept_code', right_on='code')

## Merging on Index

Use `left_index=True` and/or `right_index=True` to join on index labels.

In [ ]:
info = pd.DataFrame(
    {'city': ['Sydney', 'Melbourne', 'Brisbane']},
    index=['Alice', 'Bob', 'Charlie']
)

scores = pd.DataFrame(
    {'score': [92, 85, 78]},
    index=['Alice', 'Bob', 'Charlie']
)

pd.merge(info, scores, left_index=True, right_index=True)

In [ ]:
# Mix: merge left's column with right's index
orders_indexed = orders.set_index('customer_id')
pd.merge(customers, orders_indexed, left_on='customer_id', right_index=True)

## `.join()` — Shorthand for Index-based Merge

`.join()` is a convenience method that merges on index by default. It is equivalent to `merge` with `left_index=True, right_index=True`.

In [ ]:
info.join(scores)

In [ ]:
# join defaults to left join (unlike merge which defaults to inner)
extra_scores = pd.DataFrame(
    {'score': [92, 85, 78, 95]},
    index=['Alice', 'Bob', 'Charlie', 'Diana']
)

info.join(extra_scores, how='left')

## Handling Duplicate Column Names

When both DataFrames share column names (other than the join key), use the `suffixes` parameter to distinguish them.

In [ ]:
scores_2023 = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie'],
    'maths': [88, 76, 92],
    'english': [91, 84, 78]
})

scores_2024 = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie'],
    'maths': [90, 80, 95],
    'english': [89, 87, 82]
})

pd.merge(scores_2023, scores_2024, on='student', suffixes=('_2023', '_2024'))

## Practical Example with Data Files

In [ ]:
# Demonstrate concat with the mpg dataset split into parts
mpg = pd.read_csv('data/mpg.csv')

# Split into two halves and rejoin
first_half = mpg.iloc[:200]
second_half = mpg.iloc[200:]

rejoined = pd.concat([first_half, second_half], ignore_index=True)
print(f"Original shape: {mpg.shape}")
print(f"Rejoined shape: {rejoined.shape}")
print(f"Data matches:   {mpg.equals(rejoined)}")

## Summary

In this notebook we covered combining DataFrames:

- `pd.concat()` stacks DataFrames vertically (`axis=0`) or horizontally (`axis=1`). Use `ignore_index` and `keys` for index control.
- `pd.merge()` performs database-style joins: `inner`, `outer`, `left`, `right`.
- Use `on` when both DataFrames share the key column name; use `left_on` / `right_on` when names differ.
- Use `left_index` / `right_index` to merge on index labels.
- `.join()` is a shorthand for index-based merging (defaults to left join).
- The `suffixes` parameter resolves duplicate column names after a merge.

Next up: **Text Methods** — working with string data in pandas.